# Bayesian Inference: Titanic

This notebook shows conjugate prior, likelihood, posterior, sequential updating, prior sensitivity, Monte Carlo sampling, importance sampling, rejection sampling, and a Dirichlet-Multinomial example.

In [ ]:
from pathlib import Path
import sys
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

ROOT = Path.cwd()
if not (ROOT / "data").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

PLOTS_DIR = ROOT / "plots"
PLOTS_DIR.mkdir(exist_ok=True)

def find_data_file(file_name):
    candidates = [ROOT / file_name, ROOT / "data" / file_name]
    for path in candidates:
        if path.exists():
            return path
    raise FileNotFoundError(
        f"Could not find {file_name}. Put it in the project root or data/ folder and include the extension."
    )

def save_plot(fig, file_name):
    path = PLOTS_DIR / file_name
    fig.savefig(path, dpi=200, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved plot: {path}")
    return path

from bayesian_toolbox import BayesianToolbox

use_default_dataset = True
custom_file_name = "my_bayesian_data.csv"

data_path = ROOT / "data" / "titanic.csv" if use_default_dataset else find_data_file(custom_file_name)
tool = BayesianToolbox(data_path)
df = tool.data.copy()

success_col = "survived"
continuous_col = "age"
category_col = "pclass"
needed = [success_col, continuous_col, category_col]
missing = [c for c in needed if c not in df.columns]
if missing:
    raise ValueError(f"Set success_col/continuous_col/category_col to columns in your data. Missing: {missing}. Available: {list(df.columns)}")
if df[success_col].dropna().nunique() != 2:
    raise ValueError("Beta-Binomial example needs a binary success column.")
if pd.to_numeric(df[continuous_col], errors="coerce").dropna().empty:
    raise ValueError("Normal-Normal example needs a numeric continuous column.")

print("Dataset:", data_path)
print("Shape:", df.shape)
display(df.head())
display(df.isna().sum().to_frame("missing_values"))
display(tool.summarize_data())

In [ ]:
# Editable block: Beta-Binomial conjugate update
success_col = "survived"
success_value = 1
alpha_prior = 1
beta_prior = 1
hypothesized_p = 0.5
credibility = 0.95

result = tool.beta_binomial_from_column(success_col, success_value, alpha_prior, beta_prior, hypothesized_p, label="Titanic survival probability")
print("Likelihood:", result["likelihood"])
print("Prior:", result["prior"])
print("Posterior:", result["posterior"])
display(pd.DataFrame([result]))
display(tool.posterior_summary_beta(result, credibility))
print(f"P(p > {hypothesized_p} | data) =", tool.posterior_probability_beta(result, hypothesized_p, "greater"))
fig, ax = tool.plot_beta_prior_likelihood_posterior(result)
save_plot(fig, "bayesian_beta_prior_likelihood_posterior.png")
fig, ax = tool.plot_beta_credible_interval(result, credibility)
save_plot(fig, "bayesian_beta_credible_interval.png")

In [ ]:
# Editable block: sequential Beta-Binomial updating
values = df[success_col].dropna().eq(success_value).astype(int).to_numpy()
batches = []
for batch in np.array_split(values, 4):
    batches.append((int(batch.sum()), int(len(batch))))

seq = tool.beta_sequential_updates(batches, alpha_prior, beta_prior, hypothesized_p)
display(seq["updates"])
one_shot = result
print("Sequential final alpha,beta:", seq["updates"][["alpha_after", "beta_after"]].iloc[-1].to_dict())
print("One-shot alpha,beta:", {"alpha_posterior": one_shot["alpha_posterior"], "beta_posterior": one_shot["beta_posterior"]})
assert np.isclose(seq["updates"]["alpha_after"].iloc[-1], one_shot["alpha_posterior"])
assert np.isclose(seq["updates"]["beta_after"].iloc[-1], one_shot["beta_posterior"])
fig, ax = tool.plot_beta_sequential_updates(seq)
save_plot(fig, "bayesian_beta_sequential_updates.png")

In [ ]:
# Editable block: impact of weak and strong priors
successes = result["successes"]
trials = result["trials"]
priors = [("Weak prior Beta(1,1)", 1, 1), ("Strong prior Beta(20,20)", 20, 20)]

comparison = tool.compare_beta_priors(successes, trials, priors, hypothesized_p)
display(comparison["summary"])
fig, ax = tool.plot_beta_prior_comparison(comparison)
save_plot(fig, "bayesian_beta_weak_vs_strong_prior.png")

In [ ]:
# Editable block: Normal-Normal conjugate mean with known variance
continuous_col = "age"
prior_mean = 30
prior_variance = 100
sigma_sq = float(pd.to_numeric(df[continuous_col], errors="coerce").dropna().var(ddof=1))
hypothesized_mean = 30
credibility = 0.95

normal_result = tool.normal_mean_known_variance_update(continuous_col, prior_mean, prior_variance, sigma_sq, hypothesized_mean, label="Posterior mean age")
print("Likelihood:", normal_result["likelihood"])
print("Prior:", normal_result["prior"])
print("Posterior:", normal_result["posterior"])
display(pd.DataFrame([normal_result]).drop(columns=["data"]))
display(tool.posterior_summary_normal(normal_result, credibility))
print(f"P(mu > {hypothesized_mean} | data) =", tool.posterior_probability_normal(normal_result, hypothesized_mean, "greater"))
fig, ax = tool.plot_normal_prior_likelihood_posterior(normal_result)
save_plot(fig, "bayesian_normal_prior_likelihood_posterior.png")
fig, ax = tool.plot_normal_credible_interval(normal_result, credibility)
save_plot(fig, "bayesian_normal_credible_interval.png")

In [ ]:
# Editable block: direct Monte Carlo posterior sampling
n_samples = 10000
threshold = hypothesized_p

samples = tool.sample_beta_posterior(result, n_samples=n_samples, random_state=42)
summary = tool.monte_carlo_beta_summary(samples, threshold)
display(summary)
exact_ci = tool.posterior_summary_beta(result).set_index("quantity").loc[["credible_low", "credible_high"], "value"].to_numpy()
fig, ax = tool.plot_posterior_samples(samples, credible_interval=exact_ci)
save_plot(fig, "bayesian_monte_carlo_beta_samples.png")

In [ ]:
# Editable block: Monte Carlo convergence
exact_mean = result["alpha_posterior"] / (result["alpha_posterior"] + result["beta_posterior"])
convergence = tool.beta_monte_carlo_convergence(samples, true_value=exact_mean)
display(convergence.tail())
fig, ax = tool.plot_monte_carlo_convergence(convergence)
save_plot(fig, "bayesian_monte_carlo_convergence.png")

In [ ]:
# Editable block: importance sampling
proposal_alpha = 2
proposal_beta = 2
n_samples = 10000

importance = tool.beta_importance_sampling(successes, trials, alpha_prior, beta_prior, proposal_alpha, proposal_beta, n_samples, random_state=42)
display(pd.DataFrame([{k: v for k, v in importance.items() if k not in ["samples", "weights"]}]))
fig, ax = tool.plot_importance_sampling(importance)
save_plot(fig, "bayesian_importance_sampling_weights.png")

In [ ]:
# Editable block: rejection sampling
proposal_alpha = 1
proposal_beta = 1
n_samples = 2000

rejection = tool.beta_rejection_sampling(result, proposal_alpha, proposal_beta, n_samples=n_samples, random_state=42)
display(pd.DataFrame([{k: v for k, v in rejection.items() if k not in ["accepted_samples", "proposal_trace"]}]))
fig, ax = tool.plot_rejection_sampling(rejection)
save_plot(fig, "bayesian_rejection_sampling_trace.png")
fig, ax = tool.plot_posterior_samples(rejection["accepted_samples"])
save_plot(fig, "bayesian_rejection_sampling_accepted_samples.png")

In [ ]:
# Editable block: Dirichlet-Multinomial conjugate update
category_col = "pclass"
categories = sorted(df[category_col].dropna().unique())
alpha_prior = np.ones(len(categories))
credibility = 0.95

dirichlet_result = tool.dirichlet_multinomial_update(category_col, alpha_prior, categories, label="Titanic class probabilities")
print("Likelihood:", dirichlet_result["likelihood"])
print("Prior:", dirichlet_result["prior"])
print("Posterior:", dirichlet_result["posterior"])
display(pd.DataFrame({
    "category": dirichlet_result["categories"],
    "count": dirichlet_result["counts"],
    "alpha_prior": dirichlet_result["alpha_prior"],
    "alpha_posterior": dirichlet_result["alpha_posterior"],
}))
display(tool.posterior_summary_dirichlet(dirichlet_result, credibility))
assert np.allclose(dirichlet_result["alpha_posterior"], dirichlet_result["alpha_prior"] + dirichlet_result["counts"])
fig, ax = tool.plot_dirichlet_posterior_probabilities(dirichlet_result, credibility)
save_plot(fig, "bayesian_dirichlet_posterior_probabilities.png")
if len(categories) == 3:
    fig, ax = tool.plot_dirichlet_samples_simplex(dirichlet_result)
    save_plot(fig, "bayesian_dirichlet_simplex_samples.png")

In [ ]:
# Editable block: optional Gamma-Poisson conjugate count example
# Here counts are the number of passengers in each class in this small Titanic dataset.
counts = pd.Series(df[category_col].dropna()).value_counts().to_numpy()
alpha_prior = 1
beta_prior = 1
hypothesized_rate = counts.mean()

gamma_result = tool.gamma_poisson_update(counts, alpha_prior, beta_prior, hypothesized_rate, label="Gamma-Poisson class-count rate")
print("Likelihood:", gamma_result["likelihood"])
print("Prior:", gamma_result["prior"])
print("Posterior:", gamma_result["posterior"])
display(pd.DataFrame([gamma_result]))
display(tool.posterior_summary_gamma(gamma_result))
fig, ax = tool.plot_gamma_prior_likelihood_posterior(gamma_result)
save_plot(fig, "bayesian_gamma_poisson_posterior.png")